# Import des modules

In [1]:
import pandas as pd
import json
import re
from bs4 import BeautifulSoup

# Ouverture du Json

In [2]:
df = pd.read_json("Data.json")

df = pd.DataFrame(df)

print(df)

                         companyId contractTypes     createdAt  \
0     5Qz4LAMygzbmPiFYBS0eUoUgHIB3   [permanent]  1.717499e+15   
1     2SoSKB8DWmQ98k5qrHD6AoKnEny1   [permanent]  1.776244e+15   
2     yYSXg3ED8kNi04GBc8srtPGzWmg2   [permanent]  1.723629e+15   
3     qzoUORv54tPiZ8yQBlFU0zIMrq12   [permanent]  1.763140e+15   
4     qzoUORv54tPiZ8yQBlFU0zIMrq12   [permanent]  1.763140e+15   
...                            ...           ...           ...   
1865                           NaN   [permanent]  1.661213e+15   
1866                           NaN   [permanent]  1.682640e+15   
1867                           NaN   [permanent]  1.687219e+15   
1868                           NaN   [permanent]  1.758924e+15   
1869                           NaN   [permanent]  0.000000e+00   

                         creatorId  \
0     0eECMsCDVfTNgd3AUvKvxmIaxuQ2   
1     2SoSKB8DWmQ98k5qrHD6AoKnEny1   
2     2YSM6JomOiXbAtS1xfYmfOjpLHL2   
3     qzoUORv54tPiZ8yQBlFU0zIMrq12   
4     qzoUORv54tP

# Analyse

## On analyse la description

In [14]:
print(df['descriptionPreview'])

0       Sensefuel\nFondée en 2017 par Christophe et St...
1       Pour renforcer nos équipes, nous recherchons u...
2       Concepteur-développeur confirmé, vous disposez...
3        **Rejoins Mergify en tant que Principal Softw...
4       🎯L'entrepriseCYIM fondée en 2001, se positionn...
                              ...                        
1865    📍 Paris | Full-time | Fluent 🇫🇷 & 🇬🇧\n\n  \n\n...
1866      \n\nWinamax est une entreprise dynamique et ...
1867      \n\nWinamax est une entreprise dynamique et ...
1868    Sellsy est toujours à la recherche de personne...
1869    We are looking for a talented and motivated in...
Name: descriptionPreview, Length: 1870, dtype: str


## On analyse la description

In [25]:
print(df['details'].str['salary'].str['recurrence'].value_counts())

details
year    278
Name: count, dtype: int64


On remarque que nous n'avons que des `year`, mais certains n'ont pas de valeurs, nous allons donc tous les passer en year

# Nettoyage

## Nettoyage de la desription

In [ ]:
df['descriptionPreview'] = df['descriptionPreview'].str.replace('Description🚀', '', regex=False)
df['descriptionPreview'] = df['descriptionPreview'].str.replace('Description**', '**', regex=False)

print(df['descriptionPreview'])

In [8]:
def clean_description(text):
    if pd.isna(text) or text == "":
        return ""
    
    # on supprime les balises html 
    text = BeautifulSoup(text, "html.parser").get_text(separator=' ')
    
    # on supprime la syntaxe markdown et les liens
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    # on degage les mots en gras ou italique
    text = re.sub(r'[*_]{1,3}', '', text)
    # on supprime les titre de markdown
    text = re.sub(r'#+\s+', '', text)
    # on supprime les listes a puce
    text = re.sub(r'^\s*[-*+]\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\d+\.\s+', '', text, flags=re.MULTILINE)
    
    # 3. Nettoyage des caractères spéciaux et espaces
    text = text.replace('\n', ' ') # on supprime tout les retour a la lignr par espace
    text = re.sub(r'\s+', ' ', text) # on supprime les espaces
    
    return text.strip()

df['descriptionPreview'] = df['descriptionPreview'].apply(clean_description)

## Modification des recurrence

In [29]:
def injecter_year(row):
    if isinstance(row, dict):
        # Si salary il existe pas on le creer sous la bonne forme
        if 'salary' not in row or not isinstance(row['salary'], dict):
            row['salary'] = {}
        
        # si on a rien ou nonne alors on met `year`
        if not row['salary'].get('recurrence'):
            row['salary']['recurrence'] = 'year'
            
    return row

# on fait ca pour tout les details
df['details'] = df['details'].apply(injecter_year)


print(df['details'].str['salary'].str['recurrence'].value_counts())

details
year    1870
Name: count, dtype: int64


# Sauvegarde du json modifier

In [ ]:
df.to_json("PreProcces.json", orient="records", force_ascii=False, indent=4)